In [ ]:
import os
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import scipy as sp
import scipy.signal as signal
import torchvision.transforms.functional as tvF
from mpl_toolkits.axes_grid1 import make_axes_locatable
import math

dtype = torch.float
if torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')

train_args = {
    'alpha': 1e-1,
    'beta': 1e0,
}

# ---------------------------------------------------------
# 1. Model Definitions (TCNN)
# ---------------------------------------------------------
class TCNN(nn.Module):
    def __init__(self, TCNN_args):
        super(TCNN, self).__init__()
        self.Feature_detector = nn.Sequential(
            nn.Conv1d(in_channels=TCNN_args['in_channels'], out_channels=TCNN_args['out_channels'], kernel_size=TCNN_args['kernel_size']),
            nn.ReLU(),
        )
        self.Norm_Pool = nn.Sequential(
            nn.BatchNorm1d(num_features=TCNN_args['out_channels'], affine=False),
            nn.MaxPool1d(kernel_size=TCNN_args['maxpool_size']),
        )
        self.FC = nn.Sequential(
            nn.Flatten(start_dim=1),
            nn.Linear(in_features=TCNN_args['out_channels'] * int((TCNN_args['batch_size'] - TCNN_args['kernel_size'] + 1 - 2) / TCNN_args['maxpool_size'] + 1), out_features=TCNN_args['category_num'], bias=False),
        )

    def forward(self, x):
        return F.log_softmax(self.FC(x), dim=1)

def kernel_init_tcnn(run_TCNN, TCNN_args):
    for param in run_TCNN.Feature_detector.parameters():
        param.requires_grad = False
    run_TCNN_param = run_TCNN.state_dict()
    step = TCNN_args['kernel_size'] // 6 // TCNN_args['out_channels']
    for ii in range(0, TCNN_args['out_channels']):
        run_TCNN_param['Feature_detector.0.weight'][ii] = torch.tensor(signal.windows.gaussian(TCNN_args['kernel_size'], 2 + ii * step), dtype=dtype).to(device)
    run_TCNN_param['Feature_detector.0.bias'][:] = 0.0
    run_TCNN.load_state_dict(run_TCNN_param)
    return run_TCNN

# ---------------------------------------------------------
# 2. Model Definitions (TTE)
# ---------------------------------------------------------
class PositionalEncoding(nn.Module):
    def __init__(self, num_hiddens, dropout, max_len=1000):
        super(PositionalEncoding, self).__init__()
        self.dropout = nn.Dropout(dropout)
        self.P = torch.zeros((1, max_len, num_hiddens))
        X = torch.arange(max_len, dtype=torch.float32).reshape(-1, 1) / torch.pow(10000, torch.arange(0, num_hiddens, 2, dtype=torch.float32) / num_hiddens)
        self.P[:, :, 0::2] = torch.sin(X)
        self.P[:, :, 1::2] = torch.cos(X[:,0:2])

    def forward(self, X):
        X = X + self.P[:, :X.shape[1], :].to(X.device)
        return self.dropout(X)

class ConvEmbeding(nn.Module):
    def __init__(self, TTE_args):
        super(ConvEmbeding, self).__init__()
        self.Feature_detector = nn.Sequential(
            nn.Conv1d(in_channels=TTE_args['input_channels'], out_channels=TTE_args['expected_dim'],
                    kernel_size=TTE_args['kernel_size'], stride=TTE_args['stride']),
            nn.ReLU(),
        )
        self.Norm_Pool = nn.Sequential(
            nn.BatchNorm1d(num_features=TTE_args['expected_dim'], affine=False),
            nn.MaxPool1d(kernel_size=TTE_args['maxpool_size']),
        )
        self.Embeding = nn.Sequential(
            self.Feature_detector,
            self.Norm_Pool,
        )

    def forward(self, x):
        return self.Embeding(x).permute(0,2,1)

class TemporalTransformerEncoder(nn.Module):
    def __init__(self, TTE_args):
        super(TemporalTransformerEncoder, self).__init__()
        self.Embeding = ConvEmbeding(TTE_args)
        self.PositionalEncoding = PositionalEncoding(TTE_args['expected_dim'], TTE_args['position_dropout'])
        self.transformer_encoder = nn.TransformerEncoder(nn.TransformerEncoderLayer(TTE_args['expected_dim'], TTE_args['num_heads'], batch_first=True), TTE_args['num_layers'])
        self.FC = nn.Sequential(
            nn.Flatten(start_dim=1),
            nn.LazyLinear(out_features=TTE_args['category_num']),
        )

    def forward(self, x):
        return F.log_softmax(self.FC(self.transformer_encoder(self.PositionalEncoding(x))), dim=1)

def kernel_init_tte(run_TTE, TTE_args):
    for param in run_TTE.Embeding.Feature_detector.parameters():
        param.requires_grad = False
    run_TTE_param = run_TTE.state_dict()
    step = TTE_args['kernel_size'] // 6 // TTE_args['expected_dim']
    for ii in range(0, TTE_args['expected_dim']):
        run_TTE_param['Embeding.Feature_detector.0.weight'][ii] = torch.tensor(signal.windows.gaussian(TTE_args['kernel_size'], 2 + ii * step), dtype=dtype).to(device)
    run_TTE_param['Embeding.Feature_detector.0.bias'][:] = 0.0
    run_TTE.load_state_dict(run_TTE_param)
    return run_TTE

# ---------------------------------------------------------
# 3. Data Loading
# ---------------------------------------------------------
dat_dir = '/content/drive/MyDrive/Project/PlasticityDecoding/data'

print("Loading SPK (Burst Frc = 0.2)...")
SPK, ptype = [], []
for rnum in range(0, 30):
    SPK.append(np.load(dat_dir + f'/FixedRepresentation/Spk_an_wn_{rnum}.npy'))
    ptype.append(np.load(dat_dir + f'/FixedRepresentation/Ptype_an_wn_{rnum}.npy'))
SPK, ptype = np.concatenate(SPK, axis=0), np.concatenate(ptype, axis=0)

print("Loading SPK1 (Burst Frc = 0.4)...")
SPK1, ptype1 = [], []
for rnum in range(0, 49):
    SPK1.append(np.load(dat_dir + f'/FixedRepresentation01/Spk_an_wn_{rnum}.npy'))
    ptype1.append(np.load(dat_dir + f'/FixedRepresentation01/Ptype_an_wn_{rnum}.npy'))
SPK1, ptype1 = np.concatenate(SPK1, axis=0), np.concatenate(ptype1, axis=0)

print("Loading SPK2 (Burst Frc = 0.3)...")
SPK2, ptype2 = [], []
for rnum in range(0, 30):
    # Note: Using 02 here to prevent overlap. Adjust if files are strictly in 01 folder.
    SPK2.append(np.load(dat_dir + f'/FixedRepresentation02/Spk_an_wn_{rnum}.npy'))
    ptype2.append(np.load(dat_dir + f'/FixedRepresentation02/Ptype_an_wn_{rnum}.npy'))
SPK2, ptype2 = np.concatenate(SPK2, axis=0), np.concatenate(ptype2, axis=0)

# ---------------------------------------------------------
# 4. Helper & Training Functions
# ---------------------------------------------------------
def init_acu_dict():
    return {'acu_train': [], 'acu_test': [], 'acu_train_STDP': [], 'acu_test_STDP': [],
            'acu_train_BurstProp': [], 'acu_test_BurstProp': [], 'acu_train_BP1': [], 'acu_test_BP1': [],
            'acu_train_BP2': [], 'acu_test_BP2': [], 'acu_train_NC': [], 'acu_test_NC': []}

def acu_cal(log_py_train, y_train, log_py_test, y_test, acu):
    acu['acu_train'].append((torch.sum(torch.argmax(log_py_train, dim=1) == y_train) / y_train.size(0)).item())
    acu['acu_test'].append((torch.sum(torch.argmax(log_py_test, dim=1) == y_test) / y_test.size(0)).item())
    acu['acu_train_STDP'].append((torch.sum(torch.argmax(log_py_train, dim=1)[torch.argwhere(y_train == 0)] == y_train[torch.argwhere(y_train == 0)]) / y_train[torch.argwhere(y_train == 0)].size(0)).item())
    acu['acu_test_STDP'].append((torch.sum(torch.argmax(log_py_test, dim=1)[torch.argwhere(y_test == 0)] == y_test[torch.argwhere(y_test == 0)]) / y_test[torch.argwhere(y_test == 0)].size(0)).item())
    acu['acu_train_BurstProp'].append((torch.sum(torch.argmax(log_py_train, dim=1)[torch.argwhere(y_train == 1)] == y_train[torch.argwhere(y_train == 1)]) / y_train[torch.argwhere(y_train == 1)].size(0)).item())
    acu['acu_test_BurstProp'].append((torch.sum(torch.argmax(log_py_test, dim=1)[torch.argwhere(y_test == 1)] == y_test[torch.argwhere(y_test == 1)]) / y_test[torch.argwhere(y_test == 1)].size(0)).item())
    acu['acu_train_BP1'].append((torch.sum(torch.argmax(log_py_train, dim=1)[torch.argwhere(y_train == 2)] == y_train[torch.argwhere(y_train == 2)]) / y_train[torch.argwhere(y_train == 2)].size(0)).item())
    acu['acu_test_BP1'].append((torch.sum(torch.argmax(log_py_test, dim=1)[torch.argwhere(y_test == 2)] == y_test[torch.argwhere(y_test == 2)]) / y_test[torch.argwhere(y_test == 2)].size(0)).item())
    acu['acu_train_BP2'].append((torch.sum(torch.argmax(log_py_train, dim=1)[torch.argwhere(y_train == 3)] == y_train[torch.argwhere(y_train == 3)]) / y_train[torch.argwhere(y_train == 3)].size(0)).item())
    acu['acu_test_BP2'].append((torch.sum(torch.argmax(log_py_test, dim=1)[torch.argwhere(y_test == 3)] == y_test[torch.argwhere(y_test == 3)]) / y_test[torch.argwhere(y_test == 3)].size(0)).item())
    acu['acu_train_NC'].append((torch.sum(torch.argmax(log_py_train, dim=1)[torch.argwhere(y_train == 4)] == y_train[torch.argwhere(y_train == 4)]) / y_train[torch.argwhere(y_train == 4)].size(0)).item())
    acu['acu_test_NC'].append((torch.sum(torch.argmax(log_py_test, dim=1)[torch.argwhere(y_test == 4)] == y_test[torch.argwhere(y_test == 4)]) / y_test[torch.argwhere(y_test == 4)].size(0)).item())
    return acu

def generate_splits(train_spk_a, train_ptype_a, train_spk_b, train_ptype_b, od_spk, od_ptype):
    """Generates the merged training set and strictly held-out OOD test set"""
    SPK_train = np.concatenate([train_spk_a, train_spk_b], axis=0)
    ptype_train = np.concatenate([train_ptype_a, train_ptype_b], axis=0)

    # Strictly copied 80/20 train/test split for in-distribution
    train_ind = np.random.choice(np.size(SPK_train, axis=0), np.int32(0.8 * np.size(SPK_train, axis=0)), replace=False)
    test_ind = np.setdiff1d(np.arange(0, np.size(SPK_train, axis=0)), train_ind)

    # 20% random selection for OOD set
    test_ind_od = np.random.choice(np.size(od_spk, axis=0), np.int32(0.2 * np.size(od_spk, axis=0)), replace=False)

    x_train = torch.tensor(SPK_train[train_ind, :], dtype=dtype).to(device).unsqueeze(dim=1)
    y_train = torch.tensor(ptype_train[train_ind], dtype=torch.long).to(device)

    x_test = torch.tensor(SPK_train[test_ind, :], dtype=dtype).to(device).unsqueeze(dim=1)
    y_test = torch.tensor(ptype_train[test_ind], dtype=torch.long).to(device)

    x_test_od = torch.tensor(od_spk[test_ind_od, :], dtype=dtype).to(device).unsqueeze(dim=1)
    y_test_od = torch.tensor(od_ptype[test_ind_od], dtype=torch.long).to(device)

    return x_train, y_train, x_test, y_test, x_test_od, y_test_od

def run_tcnn_training(x_train, y_train, x_test, y_test, x_test_od, y_test_od):
    TCNN_args = {'in_channels': 1, 'out_channels': 5, 'kernel_size': 500, 'maxpool_size': 125, 'category_num': 5, 'batch_size': x_train.size(axis=2)}
    run_TCNN = TCNN(TCNN_args).to(device)
    run_TCNN = kernel_init_tcnn(run_TCNN, TCNN_args)
    loss_fn = nn.NLLLoss()
    optimizer = torch.optim.Adam(run_TCNN.parameters(), lr=1e-3)

    run_TCNN.train()
    x_train1 = run_TCNN.Norm_Pool(run_TCNN.Feature_detector(x_train))
    x_test1 = run_TCNN.Norm_Pool(run_TCNN.Feature_detector(x_test))
    x_test_od1 = run_TCNN.Norm_Pool(run_TCNN.Feature_detector(x_test_od))

    for epoch in range(0, 500):
        log_py_train = run_TCNN(x_train1)
        for weight in run_TCNN.FC.parameters():
            L1 = train_args['beta'] * F.l1_loss(torch.zeros_like(weight), weight)
        L = train_args['alpha'] * loss_fn(log_py_train, y_train) + L1
        optimizer.zero_grad()
        L.backward()
        optimizer.step()
        if torch.sum(torch.argmax(log_py_train, dim=1) == y_train) / y_train.size(0) > 0.85: break

    run_TCNN.eval()
    return (acu_cal(run_TCNN(x_train1), y_train, run_TCNN(x_test1), y_test, init_acu_dict()),
            acu_cal(run_TCNN(x_train1), y_train, run_TCNN(x_test_od1), y_test_od, init_acu_dict()))

def run_tte_training(x_train, y_train, x_test, y_test, x_test_od, y_test_od):
    TTE_args = {'input_channels': 1, 'kernel_size': 500, 'stride': 1, 'maxpool_size': 125, 'num_heads': 5, 'expected_dim': 5, 'num_layers': 1, 'category_num': 5, 'position_dropout': 0.0, 'data_len': x_train.size(dim=1)}
    run_TTE = TemporalTransformerEncoder(TTE_args).to(device)
    run_TTE = kernel_init_tte(run_TTE, TTE_args)
    loss_fn = nn.NLLLoss()
    optimizer = torch.optim.Adam(run_TTE.parameters(), lr=1e-3)

    run_TTE.train()
    x_train1 = run_TTE.Embeding(x_train)
    x_test1 = run_TTE.Embeding(x_test)
    x_test_od1 = run_TTE.Embeding(x_test_od)

    for epoch in range(0, 500):
        log_py_train = run_TTE(x_train1)
        L = loss_fn(log_py_train, y_train)
        optimizer.zero_grad()
        L.backward()
        optimizer.step()
        if torch.sum(torch.argmax(log_py_train, dim=1) == y_train) / y_train.size(0) > 0.85: break

    run_TTE.eval()
    return (acu_cal(run_TTE(x_train1), y_train, run_TTE(x_test1), y_test, init_acu_dict()),
            acu_cal(run_TTE(x_train1), y_train, run_TTE(x_test_od1), y_test_od, init_acu_dict()))

def parse_acu(acu_dict):
    """Flattens the acu dictionary into a 12-element list for matrix assignment"""
    return [acu_dict['acu_train'][0], acu_dict['acu_test'][0], acu_dict['acu_train_STDP'][0], acu_dict['acu_test_STDP'][0],
            acu_dict['acu_train_BurstProp'][0], acu_dict['acu_test_BurstProp'][0], acu_dict['acu_train_BP1'][0], acu_dict['acu_test_BP1'][0],
            acu_dict['acu_train_BP2'][0], acu_dict['acu_test_BP2'][0], acu_dict['acu_train_NC'][0], acu_dict['acu_test_NC'][0]]


# ---------------------------------------------------------
# 5. Main Execution Loop (50 iterations)
# ---------------------------------------------------------
# Matrices for Test 1: Train (0.2, 0.3) -> Test OOD (0.4)
tcnn_in_1, tcnn_out_1, tte_in_1, tte_out_1 = np.zeros((50, 12)), np.zeros((50, 12)), np.zeros((50, 12)), np.zeros((50, 12))
# Matrices for Test 2: Train (0.2, 0.4) -> Test OOD (0.3)
tcnn_in_2, tcnn_out_2, tte_in_2, tte_out_2 = np.zeros((50, 12)), np.zeros((50, 12)), np.zeros((50, 12)), np.zeros((50, 12))
# Matrices for Test 3: Train (0.3, 0.4) -> Test OOD (0.2)
tcnn_in_3, tcnn_out_3, tte_in_3, tte_out_3 = np.zeros((50, 12)), np.zeros((50, 12)), np.zeros((50, 12)), np.zeros((50, 12))

print("Starting 50 Iterations of Broad OOD Testing...")
for ii in range(50):
    # --- Test 1: Train (SPK + SPK2) -> Test OOD (SPK1) ---
    x_train, y_train, x_test, y_test, x_od, y_od = generate_splits(SPK, ptype, SPK2, ptype2, SPK1, ptype1)

    acu_in_tcnn, acu_out_tcnn = run_tcnn_training(x_train, y_train, x_test, y_test, x_od, y_od)
    tcnn_in_1[ii, :], tcnn_out_1[ii, :] = parse_acu(acu_in_tcnn), parse_acu(acu_out_tcnn)

    acu_in_tte, acu_out_tte = run_tte_training(x_train, y_train, x_test, y_test, x_od, y_od)
    tte_in_1[ii, :], tte_out_1[ii, :] = parse_acu(acu_in_tte), parse_acu(acu_out_tte)

    # --- Test 2: Train (SPK + SPK1) -> Test OOD (SPK2) ---
    x_train, y_train, x_test, y_test, x_od, y_od = generate_splits(SPK, ptype, SPK1, ptype1, SPK2, ptype2)

    acu_in_tcnn, acu_out_tcnn = run_tcnn_training(x_train, y_train, x_test, y_test, x_od, y_od)
    tcnn_in_2[ii, :], tcnn_out_2[ii, :] = parse_acu(acu_in_tcnn), parse_acu(acu_out_tcnn)

    acu_in_tte, acu_out_tte = run_tte_training(x_train, y_train, x_test, y_test, x_od, y_od)
    tte_in_2[ii, :], tte_out_2[ii, :] = parse_acu(acu_in_tte), parse_acu(acu_out_tte)

    # --- Test 3: Train (SPK1 + SPK2) -> Test OOD (SPK) ---
    x_train, y_train, x_test, y_test, x_od, y_od = generate_splits(SPK1, ptype1, SPK2, ptype2, SPK, ptype)

    acu_in_tcnn, acu_out_tcnn = run_tcnn_training(x_train, y_train, x_test, y_test, x_od, y_od)
    tcnn_in_3[ii, :], tcnn_out_3[ii, :] = parse_acu(acu_in_tcnn), parse_acu(acu_out_tcnn)

    acu_in_tte, acu_out_tte = run_tte_training(x_train, y_train, x_test, y_test, x_od, y_od)
    tte_in_3[ii, :], tte_out_3[ii, :] = parse_acu(acu_in_tte), parse_acu(acu_out_tte)

# ---------------------------------------------------------
# 6. Saving and Output
# ---------------------------------------------------------
save_dir = '/content/drive/MyDrive/Project/PlasticityDecoding/result_20260601'
os.makedirs(save_dir, exist_ok=True)

# Save TCNN
np.save(f'{save_dir}/TCNN_broad_OOD_Test1_in.npy', tcnn_in_1); np.save(f'{save_dir}/TCNN_broad_OOD_Test1_out.npy', tcnn_out_1)
np.save(f'{save_dir}/TCNN_broad_OOD_Test2_in.npy', tcnn_in_2); np.save(f'{save_dir}/TCNN_broad_OOD_Test2_out.npy', tcnn_out_2)
np.save(f'{save_dir}/TCNN_broad_OOD_Test3_in.npy', tcnn_in_3); np.save(f'{save_dir}/TCNN_broad_OOD_Test3_out.npy', tcnn_out_3)

# Save TTE
np.save(f'{save_dir}/TTE_broad_OOD_Test1_in.npy', tte_in_1); np.save(f'{save_dir}/TTE_broad_OOD_Test1_out.npy', tte_out_1)
np.save(f'{save_dir}/TTE_broad_OOD_Test2_in.npy', tte_in_2); np.save(f'{save_dir}/TTE_broad_OOD_Test2_out.npy', tte_out_2)
np.save(f'{save_dir}/TTE_broad_OOD_Test3_in.npy', tte_in_3); np.save(f'{save_dir}/TTE_broad_OOD_Test3_out.npy', tte_out_3)

print("\n=======================================================")
print("Results: Broad OOD Generalization (In-Dist / Out-Dist)")
print("=======================================================")

tests = [
    ("Test 1: Train (0.2, 0.3) -> Test OOD (0.4)", tcnn_in_1, tcnn_out_1, tte_in_1, tte_out_1),
    ("Test 2: Train (0.2, 0.4) -> Test OOD (0.3)", tcnn_in_2, tcnn_out_2, tte_in_2, tte_out_2),
    ("Test 3: Train (0.4, 0.3) -> Test OOD (0.2)", tcnn_in_3, tcnn_out_3, tte_in_3, tte_out_3)
]

for name, tcnn_in, tcnn_out, tte_in, tte_out in tests:
    tm_in = np.round(tcnn_in.mean(axis=0), 3)
    tm_out = np.round(tcnn_out.mean(axis=0), 3)
    em_in = np.round(tte_in.mean(axis=0), 3)
    em_out = np.round(tte_out.mean(axis=0), 3)

    print(f"\n{name}")
    print(f"TCNN Overall (Train/Test) : In-Dist: {tm_in[0]:.3f}/{tm_in[1]:.3f} | Out-Dist: {tm_out[0]:.3f}/{tm_out[1]:.3f}")
    print(f"TCNN STDP                 : In-Dist: {tm_in[2]:.3f}/{tm_in[3]:.3f} | Out-Dist: {tm_out[2]:.3f}/{tm_out[3]:.3f}")
    print(f"TCNN BDP                  : In-Dist: {tm_in[4]:.3f}/{tm_in[5]:.3f} | Out-Dist: {tm_out[4]:.3f}/{tm_out[5]:.3f}")
    print(f"TCNN BPTT1                : In-Dist: {tm_in[6]:.3f}/{tm_in[7]:.3f} | Out-Dist: {tm_out[6]:.3f}/{tm_out[7]:.3f}")
    print(f"TCNN BPTT2                : In-Dist: {tm_in[8]:.3f}/{tm_in[9]:.3f} | Out-Dist: {tm_out[8]:.3f}/{tm_out[9]:.3f}")
    print(f"TCNN NC                   : In-Dist: {tm_in[10]:.3f}/{tm_in[11]:.3f} | Out-Dist: {tm_out[10]:.3f}/{tm_out[11]:.3f}")

    print(f"\nTTE  Overall (Train/Test) : In-Dist: {em_in[0]:.3f}/{em_in[1]:.3f} | Out-Dist: {em_out[0]:.3f}/{em_out[1]:.3f}")
    print(f"TTE  STDP                 : In-Dist: {em_in[2]:.3f}/{em_in[3]:.3f} | Out-Dist: {em_out[2]:.3f}/{em_out[3]:.3f}")
    print(f"TTE  BDP                  : In-Dist: {em_in[4]:.3f}/{em_in[5]:.3f} | Out-Dist: {em_out[4]:.3f}/{em_out[5]:.3f}")
    print(f"TTE  BPTT1                : In-Dist: {em_in[6]:.3f}/{em_in[7]:.3f} | Out-Dist: {em_out[6]:.3f}/{em_out[7]:.3f}")
    print(f"TTE  BPTT2                : In-Dist: {em_in[8]:.3f}/{em_in[9]:.3f} | Out-Dist: {em_out[8]:.3f}/{em_out[9]:.3f}")
    print(f"TTE  NC                   : In-Dist: {em_in[10]:.3f}/{em_in[11]:.3f} | Out-Dist: {em_out[10]:.3f}/{em_out[11]:.3f}")

print("\nDone! Full class-by-class metrics exported to Google Drive.")

Loading SPK (Burst Frc = 0.2)...
Loading SPK1 (Burst Frc = 0.4)...
Loading SPK2 (Burst Frc = 0.3)...
Starting 50 Iterations of Broad OOD Testing...


/tmp/ipykernel_4468/127193466.py:100: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.num_heads is odd
  self.transformer_encoder = nn.TransformerEncoder(nn.TransformerEncoderLayer(TTE_args['expected_dim'], TTE_args['num_heads'], batch_first=True), TTE_args['num_layers'])



Results: Broad OOD Generalization (In-Dist / Out-Dist)

Test 1: Train (0.2, 0.3) -> Test OOD (0.4)
TCNN Overall (Train/Test) : In-Dist: 0.787/0.733 | Out-Dist: 0.787/0.695
TCNN STDP                 : In-Dist: 0.877/0.837 | Out-Dist: 0.877/0.805
TCNN BDP                  : In-Dist: 0.703/0.613 | Out-Dist: 0.703/0.585
TCNN BPTT1                : In-Dist: 0.607/0.532 | Out-Dist: 0.607/0.426
TCNN BPTT2                : In-Dist: 0.777/0.726 | Out-Dist: 0.777/0.716
TCNN NC                   : In-Dist: 0.972/0.952 | Out-Dist: 0.972/0.942

TTE  Overall (Train/Test) : In-Dist: 0.862/0.787 | Out-Dist: 0.862/0.735
TTE  STDP                 : In-Dist: 0.943/0.899 | Out-Dist: 0.943/0.831
TTE  BDP                  : In-Dist: 0.851/0.736 | Out-Dist: 0.851/0.723
TTE  BPTT1                : In-Dist: 0.710/0.596 | Out-Dist: 0.710/0.438
TTE  BPTT2                : In-Dist: 0.822/0.735 | Out-Dist: 0.822/0.713
TTE  NC                   : In-Dist: 0.984/0.962 | Out-Dist: 0.984/0.975

Test 2: Train (0.2, 0.